# Furstenberg-Sárközy Problem

In [ ]:
#@title Verification code

# pylint: disable=unused-import
# pylint: disable=g-bad-import-order



def is_square_free(n):
  """Checks if an integer n is square-free.

  Args:
      n (int): The integer to check.

  Returns:
      bool: True if n is square-free, False otherwise.
  """
  if n <= 1:
    return True  # 1 is considered square-free

  # Handle negative numbers (square-freeness is usually defined for positive
  # integers)
  if n < 0:
    n = -n

  # Check divisibility by squares of primes up to sqrt(n)
  d = 2
  while d * d <= n:
    if n % (d * d) == 0:
      return False
    d += 1
  return True


def get_score(best_list, best_m):
  """Returns the score for the best list."""
  # if best_m is not square-free, return 0
  if not is_square_free(best_m):
    return 0
  best_list_set = set(best_list)
  # reduce each element mod best_m
  best_list_set = {x % best_m for x in best_list_set}
  kth_power_residues = set()
  for y in range(1, best_m):
    residue = pow(y, 2, best_m)
    if residue != 0:
      kth_power_residues.add(residue)
  non_zero_residues = sorted(list(kth_power_residues))

  # Verify that no two elements differ by a non-zero k-th power residue
  for i in best_list_set:
    for j in best_list_set:
      if (i - j) % best_m in non_zero_residues:
        return 0

  return 1 - 1.0 / 2.0 + math.log(len(best_list_set)) / (2 * math.log(best_m))


def format_feedback_repr(feedback):
  """Formats feedback dictionary for representation in code."""
  formatted_feedback = {}
  for key, value in feedback.items():
    if isinstance(value, np.ndarray):
      repr_str = repr(value)  # Get repr string (e.g., "array([[...], [...]])")
      cleaned_repr_str = re.sub(r'[\n\s]+', ' ', repr_str)  # Clean up

      # Remove the leading "array(" and trailing ")" from repr string, then wrap
      # with "np.array(...)"
      array_content = cleaned_repr_str[
          6:-1
      ]  # Extract content inside "array(...)"

      if np.iscomplexobj(value):
        formatted_feedback[key] = (  # Use extracted content in np.array
            f'np.array({array_content}, dtype=np.complex128)'
        )
      elif not np.issubdtype(value.dtype, np.inexact):
        formatted_feedback[key] = f'np.array({array_content}, dtype=np.float64)'
      else:
        formatted_feedback[key] = f'np.array({array_content})'

    elif isinstance(value, list):
      formatted_feedback[key] = repr(value)  # Use standard repr for lists
    else:
      formatted_feedback[key] = repr(value)
  return formatted_feedback


def evaluate(
    hypers: Mapping[str, Any],
) -> tuple[dict[str, float], dict[str, str]]:
  """Evaluates the program and returns the score and feedback."""
  result = {}
  feedback = {}
  del hypers
  best_list, best_m = find_best_set_furstenberg()
  best_list = [x % best_m for x in best_list]
  best_list = sorted(list(set(best_list)))
  score = get_score(best_list, best_m)

  result['score'] = score
  feedback['best_list'] = best_list
  feedback['best_score_found'] = score
  feedback['best_m'] = best_m
  feedback = format_feedback_repr(feedback)
  return result, feedback

In [ ]:
#@title Initial program

"""Finds joint probability distributions to test Sum-Difference Conjecture."""
import itertools
import logging
import time
from scipy import integrate
import numpy as np
from scipy import optimize
import warnings
import math
import re
from typing import Any, Callable, Mapping, List, Tuple
import scipy.linalg as la
import numpy.polynomial.polynomial as poly
import collections
from google3.util.operations_research.math_opt.python import mathopt
import numba

njit = numba.njit
minimize = optimize.minimize





def find_best_set_furstenberg():
  """Finds the best set of numbers modulo m with no k-th power difference."""
  best_list = largest_subset_no_kth_power_difference(205)
  best_m = 22
  if best_list:
    pass
  else:
    best_list = []
  best_score = get_score(best_list, best_m)
  eval_count = 0
  start_time = time.time()
  while time.time() - start_time < 10:  # Search for 1000 seconds
    curr_m = np.random.randint(1, 200)
    if not is_square_free(curr_m):
      continue
    curr_list = largest_subset_no_kth_power_difference(curr_m)
    if curr_list is None:
      continue
    score = get_score(curr_list, curr_m)
    eval_count += 1
    if score > best_score:
      best_score = score
      best_list = curr_list.copy()
      best_m = curr_m
      print(f'Best score: {score}')
      print(set(best_list))
  return best_list, best_m


def largest_subset_no_kth_power_difference(m, k=2):
  """Finds the largest subset of Z/mZ with no k-th power residue difference.

  Args:
      m (int): The modulus.
      k (int): The power.

  Returns:
      set: The largest subset of Z/mZ.
  """

  # 1. Find the set of non-zero k-th power residues modulo m
  kth_power_residues = set()
  for y in range(1, m):
    residue = pow(y, k, m)
    if residue != 0:
      kth_power_residues.add(residue)
  non_zero_residues = sorted(list(kth_power_residues))

  # 2. Create the mathopt model
  model = mathopt.Model(name=f'LargestSubset_m{m}_k{k}')

  # 3. Define the binary variables
  x = [
      model.add_integer_variable(lb=0.0, ub=1.0, name=f'x_{i}')
      for i in range(m)
  ]

  # 4. Set the objective function
  objective_expr = 0
  for var in x:
    objective_expr += var
  model.maximize(objective_expr)

  # 5. Add the constraints
  for i in range(m):
    for j in range(i + 1, m):
      diff1 = (i - j) % m
      diff2 = (j - i) % m
      for r in non_zero_residues:
        if diff1 == r or diff2 == r:
          model.add_linear_constraint(
              x[i] + x[j] <= 1.0, name=f'constraint_{i}_{j}_{r}'
          )

  # 6. Solve the IP problem
  try:
    result = mathopt.solve(
        model, mathopt.SolverType.GSCIP
    )  # Assuming GSCIP is available
  except RuntimeError as e:
    print(f'Error during solve: {e}')
    return None

  # 7. Extract and return the result (the set)
  if result.termination.reason in (
      mathopt.TerminationReason.OPTIMAL,
      mathopt.TerminationReason.FEASIBLE,
  ):
    largest_subset = {
        i for i in range(m) if round(result.variable_values()[x[i]]) == 1
    }
    return list(largest_subset)
  else:
    print(f'Solver status: {result.termination}')
    return []

**Prompt used**

Act as an expert software developer and optimization specialist specializing in
creating python lists of distinct integers with certain properties. Your task is to find a python list of distinct integers that is the largest subset of 1,...,N that does not contain any two elements that differ by a perfect cube modulo N. The key here will be to choose N wisely. It has to be square-free, but otherwise not much is known about what N is a good candidate.
The exact score function your construction will be evaluated on is given below:

def get_score(best_list, best_m):
  """Returns the score for the best list."""
  # if best_m is not square-free, return 0

if not is_square_free(best_m):
    return 0
  best_list_set = set(best_list)
  best_list_set = set([x % best_m for x in best_list_set])
  kth_power_residues = set()
  for y in range(1, best_m):
    residue = pow(y, 3, best_m)
    if residue != 0:
      kth_power_residues.add(residue)
  non_zero_residues = sorted(list(kth_power_residues))

## What AlphaEvolve found

AlphaEvolve was tasked with finding subsets of $\mathbb{Z}/m\mathbb{Z}$ avoiding square (resp. cube) differences, aiming to improve the lower bounds for $C(2, N)$ and $C(3, N)$. AlphaEvolve managed to quickly reproduce the known lower bounds — a 12-element subset of $\mathbb{Z}/205\mathbb{Z}$ avoiding square differences, and a 14-element subset of $\mathbb{Z}/91\mathbb{Z}$ avoiding cube differences — but it did not find anything better.